In [2]:
import pandas as pd
import numpy as np
import glob
from scipy import stats
import re
from datetime import datetime
from collections import Counter

import matplotlib.pyplot as plt

In [10]:
from pathlib import Path
import yaml

CONFIG_PATH = Path("../workflow/config.yaml")

with CONFIG_PATH.open("r") as f:
    config = yaml.safe_load(f)

In [13]:
SYNC_DIR = config["sync_dir"]

filelist = glob.glob(f"{SYNC_DIR}/SYNC_network*")

In [14]:
find_number = re.compile("network[0-9]+")
find_type = re.compile("_[a-z]+")
N = len(filelist)

stats_mean_g = np.zeros(int(N/4))
stats_sd_g = np.zeros(int(N/4))
stats_mean_m = np.zeros(int(N/4))
stats_sd_m = np.zeros(int(N/4))
stats_mean = np.zeros(int(N/4))
stats_sd = np.zeros(int(N/4))
stats_mean_c = np.zeros(int(N/4))
stats_sd_c = np.zeros(int(N/4))

for filename in filelist:
    rank = pd.read_csv(filename)
    when = int(re.findall("\d+", filename)[1])
#     when = int(re.findall("\d+", filename)[0])
#     type_ = find_type.search(filename).group()
    type_ = re.findall(find_type, filename)[-1]

    if type_ == "_genesis":
        stats_mean_g[when] = rank["2"].mean()
        stats_sd_g[when] = stats.sem(rank["2"])
    elif type_ == "_miner":
        stats_mean_m[when] = rank["2"].mean()
        stats_sd_m[when] = stats.sem(rank["2"])
    elif type_ == "_random":
        stats_mean[when] = rank["2"].mean()
        stats_sd[when] = stats.sem(rank["2"])
    elif type_ == "_cross":
        stats_mean_c[when] = rank["2"].mean()
        stats_sd_c[when] = stats.sem(rank["2"])

In [15]:
stats_correlation = pd.DataFrame()

stats_correlation["genesis_mean"] = stats_mean_g
stats_correlation["genesis_sd"] = stats_sd_g
stats_correlation["miner_mean"] = stats_mean_m
stats_correlation["miner_sd"] = stats_sd_m
stats_correlation["cross_mean"] = stats_mean_c
stats_correlation["cross_sd"] = stats_sd_c
stats_correlation["random_mean"] = stats_mean
stats_correlation["random_sd"] = stats_sd

In [ ]:
# normalization

for col in ["genesis_mean", "miner_mean", "cross_mean"]:
    stats_correlation["normalized_"+col] = stats_correlation.apply(lambda x: x[col]/x["random_mean"], axis=1)


cols = ["genesis_mean", "miner_mean", "cross_mean", "random_mean"]
cols = ["normalized_genesis_mean", "normalized_miner_mean"] #, "normalized_cross_mean"]
colors = ["cornflowerblue","salmon", "orchid", "gray"]

date_idx = pd.date_range(start='08/01/2015', end='12/31/2016', freq='1MS') 

plt.rcParams["font.family"]="Arial"
plt.rcParams["font.size"] = 20
# plt.rcParams["text.latex.preamble"] = [r"\usepackage{bm}"]                                       
fig, ax = plt.subplots(1, figsize=(12,9))

for i, col in enumerate(cols):
    name = col.split("_")[0] + "_sd"
    # error = stats_correlation[name]
# #     ax.errorbar(date_idx, stats_correlation[col], yerr=error, linestyle='')
    ax.plot(date_idx, stats_correlation[col], marker="o", mfc="none", label=col.split("_")[1], markersize=10, markeredgewidth=3, linewidth=3, color=colors[i])
    # ax.fill_between(date_idx, stats_correlation[col] - error, stats_correlation[col] + error, alpha=0.25, color=colors[i])
    
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
# ax.axvline(x=datetime(2016, 7, 20), color='r', linestyle=':', linewidth=3)
ax.axhline(y=1, color='gray', linewidth=2)
# ax.text(datetime(2016, 7, 7),0.94, "2016.07\nHardFork", fontsize=30)
ax.set_xlabel("Month")
# ax.set_ylabel(r"$<\mathbf{\tau}>$", fontsize=25)
ax.set_ylabel(r"$<{\tau}>/<{\tau_{rand}}>$", fontsize=25)
# ax.tick_params(axis='x', labelrotation=20)
ax.legend()

showdate = date_idx[[0,2,4,6,8,10,12,14,16]]
plt.xticks(showdate, 
           [ts.strftime('%b\n%Y') if ts.year != showdate[idx-1].year
                    else ts.strftime('%b') for idx, ts in enumerate(showdate)])

plt.axvline(x=date_idx[8], color='gray', linestyle=':', linewidth=5)
plt.axvline(x=date_idx[11], color='gray', linestyle=':', linewidth=5)


plt.savefig("../figures/rank_correlation.pdf", bbox_inches='tight', dpi=200)

In [ ]:
import matplotlib.dates as mdates

plt.rcParams["figure.figsize"]=(5,4)
plt.rcParams["font.size"]=15
# plt.axvline(x=datetime(2016, 7, 20), color='r', linestyle=':', linewidth=3)
plt.plot(date_idx, stats_correlation["random_mean"], marker="o", mfc="none", markersize=10, markeredgewidth=3, linewidth=3, color="gray")
error = stats_correlation["random_sd"]
plt.fill_between(date_idx, stats_correlation["random_mean"] - error, stats_correlation["random_mean"] + error, alpha=0.25, color="gray")
    

plt.ylabel(r"$<{\tau_{rand}}>$", fontsize=25)
# monthyearFmt = mdates.DateFormatter('%b')
# plt.gca().xaxis.set_major_formatter(monthyearFmt)
showdate = [date_idx[0], date_idx[3], date_idx[6], date_idx[9], date_idx[12], date_idx[15]]
plt.xticks(showdate, 
           [ts.strftime('%b\n%Y') if ts.year != showdate[idx-1].year
                    else ts.strftime('%b') for idx, ts in enumerate(showdate)])

plt.axvline(x=date_idx[8], color='gray', linestyle=':', linewidth=5)
plt.axvline(x=date_idx[11], color='gray', linestyle=':', linewidth=5)

plt.savefig("../figures/random.pdf", bbox_inches="tight")